# Обучение моделей

In [104]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.precision', 4)

In [105]:
from pathlib import Path

path_p1_p2 = Path("..", "data", "processed", "p1_p2_data.csv")
df_p1_p2 = pd.read_csv(path_p1_p2, sep=";")

path_p3 = Path("..", "data", "processed", "p3_data.csv")
df_p3 = pd.read_csv(path_p3, sep=";")

path_altpay5 = Path("..", "data", "processed", "altpay5_data.csv")
df_altpay5 = pd.read_csv(path_altpay5, sep=";")

dfs = [df_p1_p2, df_p3, df_altpay5]

COMMON_DTTM_COLS = [
    'month',
    'first_trx_month_inn',
    'last_active_month_inn',
]

CAT_COLS = [
    'inn_status',
    'top_mcc_group_inn',
    'who_is_this_first_inn',
]

for df in dfs:
    for col in COMMON_DTTM_COLS:
        df[col] = pd.to_datetime(df[col])

    # В каждом df разный набор колонок с датами подключения других продуктов
    first_month_cols = [col for col in df.columns if col.startswith("first_month")]
    for col in first_month_cols:
        df[col] = pd.to_datetime(df[col])

    for col in CAT_COLS:
        df[col] = df[col].astype('category')

## Метрики качества
Основной метрикой качества моделей является Precision@K, поскольку бизнес-задача заключается в эффективном распределении ограниченного ресурса отдела продаж. В рамках задачи capacity отдела составляет 2000 клиентов в месяц, что определяет основное значение K. Дополнительно рассматриваются значения K=500, K=1000 и K=1500 для анализа устойчивости качества ранжирования.

Для оценки относительного качества моделей используется метрика Uplift — отношение Precision@K рассматриваемой модели к Precision@K базовых подходов. В качестве базовых подходов используются:

1. Случайный отбор клиентов
2. Отбор на основе текущих бизнес-правил

Дополнительно для оценки качества ранжирования анализируется зависимость Precision@K от K, что позволяет оценить способность модели выделять наиболее релевантных клиентов в верхней части ранжированного списка.

Также используется метрика Recall@K, отражающая долю всех целевых событий (подключений продукта), попадающих в top-K клиентов, что позволяет оценить полноту охвата моделью потенциально релевантных клиентов.

Для интерпретации результатов в бизнес-терминах рассчитывается ожидаемое количество подключений (Expected Conversions), определяемое как произведение Precision@K на K.

## P1P2

### Случайный отбор


Для случайной модели ранжирования ожидаемое значение Precision@K совпадает с долей положительных объектов в выборке. В связи с этим в качестве baseline используется доля положительного класса, рассчитанная по каждому месяцу.

In [106]:
k_values = [500, 1000, 1500, 2000, 2500]

precision_k = (
    df_p1_p2.groupby('month').target.sum() 
    / df_p1_p2.groupby('month').size()
).mean()

print("Random Selection P1P2")

for k in k_values:
    recall_k = (df_p1_p2.groupby('month').apply(lambda x: k / len(x))).mean()
    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {precision_k * k:.2f}"
    )

Random Selection P1P2
K=500  | Precision: 0.23% | Recall: 2.23% | Expected Conversions: 1.15
K=1000 | Precision: 0.23% | Recall: 4.45% | Expected Conversions: 2.30
K=1500 | Precision: 0.23% | Recall: 6.68% | Expected Conversions: 3.45
K=2000 | Precision: 0.23% | Recall: 8.91% | Expected Conversions: 4.59
K=2500 | Precision: 0.23% | Recall: 11.13% | Expected Conversions: 5.74


Качество случайного отбора ожидаемо неустойчиво по месяцам. Возможно, подключения имеют выраженную сезонность, из-за чего во втором полугодии наблюдается более высокая доля подключений среди всех наблюдений в сравнении с первым полугодием.

### Business-Rule

Критерии релевантности:
- Отсутствие подключённых продуктов `P1`, `P2` (фильтрация уже реализована в `df_p1_p2`)
- Статус `'Active'`, `'Reborn'`
- Группа MCC `!= 'Charity'`

Кандидаты на подключение ранжируется по убыванию `turnover`

In [107]:
def add_rule_flag_p1_p2(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]
    banned_mcc_groups = ["Charity"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & ~df["top_mcc_group_inn"].isin(banned_mcc_groups)
    ).astype("int8")

    return df


df_p1_p2 = add_rule_flag_p1_p2(df_p1_p2)

Замерим качество.

In [108]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'turnover'

print(f"Business-Rule Selection P1P2, y_score='{y_score}'")

for k in k_values:
    df_k = (
        df_p1_p2.query('rule_flag == 1')
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        df_p1_p2[df_p1_p2['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Business-Rule Selection P1P2, y_score='turnover'
K=500  | Precision: 0.56% | Recall: 5.67% | Expected Conversions: 2.80
K=1000 | Precision: 0.55% | Recall: 11.33% | Expected Conversions: 5.53
K=1500 | Precision: 0.52% | Recall: 16.06% | Expected Conversions: 7.80
K=2000 | Precision: 0.53% | Recall: 21.24% | Expected Conversions: 10.60
K=2500 | Precision: 0.53% | Recall: 26.61% | Expected Conversions: 13.33


## P3

### Случайный отбор

In [109]:
k_values = [500, 1000, 1500, 2000, 2500]

precision_k = (
    df_p3.groupby('month').target.sum()
    / df_p3.groupby('month').size()
).mean()

print("Random Selection P3")

for k in k_values:
    recall_k = (df_p3.groupby('month').apply(lambda x: k / len(x))).mean()
    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {precision_k * k:.2f}"
    )

Random Selection P3
K=500  | Precision: 0.01% | Recall: 1.94% | Expected Conversions: 0.03
K=1000 | Precision: 0.01% | Recall: 3.87% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Recall: 5.81% | Expected Conversions: 0.10
K=2000 | Precision: 0.01% | Recall: 7.74% | Expected Conversions: 0.14
K=2500 | Precision: 0.01% | Recall: 9.68% | Expected Conversions: 0.17


### Business-Rule

Критерии релевантности:
- Отсутствие подключённого продукта `P3`
- Статус `'Active'`, `'Reborn'`
- Релевантные MCC (флаг `is_relevant_mcc_p3`)
- Группа MCC `!= 'Charity'`

In [110]:
def add_rule_flag_p3(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]
    banned_mcc_groups = ["Charity"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & ~df["top_mcc_group_inn"].isin(banned_mcc_groups)
        & df["is_relevant_mcc_p3"]
    ).astype("int8")

    return df


df_p3 = add_rule_flag_p3(df_p3)

In [111]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'turnover'

print(f"Business-Rule Selection P3, y_score='{y_score}'")

for k in k_values:
    df_k = (
        df_p3.query('rule_flag == 1')
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        df_p3[df_p3['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Business-Rule Selection P3, y_score='turnover'
K=500  | Precision: 0.03% | Recall: 7.58% | Expected Conversions: 0.13
K=1000 | Precision: 0.02% | Recall: 10.61% | Expected Conversions: 0.20
K=1500 | Precision: 0.01% | Recall: 10.61% | Expected Conversions: 0.20
K=2000 | Precision: 0.02% | Recall: 28.79% | Expected Conversions: 0.47
K=2500 | Precision: 0.02% | Recall: 31.82% | Expected Conversions: 0.53


## Altpay5

### Случайный отбор

In [112]:
k_values = [500, 1000, 1500, 2000]

precision_k = (
    df_altpay5.groupby('month').target.sum()
    / df_altpay5.groupby('month').size()
).mean()

print("Random Selection Altpay5")

for k in k_values:
    recall_k = df_altpay5.groupby('month').apply(lambda x: k / len(x))
    print(
        f"K={k:<4} | "
        f"Precision: {precision_k.mean():.2%} | "
        f"Recall: {recall_k.mean():.2%} | "
        f"Expected Conversions: {precision_k.mean() * k:.2f}"
    )

Random Selection Altpay5
K=500  | Precision: 0.14% | Recall: 1.97% | Expected Conversions: 0.68
K=1000 | Precision: 0.14% | Recall: 3.94% | Expected Conversions: 1.35
K=1500 | Precision: 0.14% | Recall: 5.91% | Expected Conversions: 2.03
K=2000 | Precision: 0.14% | Recall: 7.88% | Expected Conversions: 2.70


### Business-Rule

Критерии релевантности:
- Отсутствие подключённого продукта `Altpay5`
- Статус `'Active'`, `'Reborn'`
- Только релевантные MCC (флаг `is_relevant_mcc_altpay5`)
- Минимальный порог оборота (флаг `is_relevant_turnover_altpay5`)

In [113]:
def add_rule_flag_altpay5(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & df["is_relevant_mcc_altpay5"]
        & df["is_relevant_turnover_altpay5"]
    ).astype("int8")

    return df


df_altpay5 = add_rule_flag_altpay5(df_altpay5)

In [114]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'turnover'

print(f"Business-Rule Selection Altpay5, y_score='{y_score}'")

for k in k_values:
    df_k = (
        df_altpay5.query('rule_flag == 1')
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        df_altpay5[df_altpay5['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Business-Rule Selection Altpay5, y_score='turnover'
K=500  | Precision: 1.00% | Recall: 16.40% | Expected Conversions: 5.00
K=1000 | Precision: 0.90% | Recall: 27.94% | Expected Conversions: 9.00
K=1500 | Precision: 0.79% | Recall: 36.16% | Expected Conversions: 11.80
K=2000 | Precision: 0.76% | Recall: 45.31% | Expected Conversions: 15.20
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29
